In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from catboost import CatBoostRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
df = pd.read_csv("오월드.csv", encoding="utf-8-sig")

feature_cols = ["휴일구분", "평균기온", "일강수량", "평균상대습도"]
target_col = "전체건수"

#구간 범위 지정
bins = [0, 700, 1500, np.inf]
labels = [0, 1, 2]

X = df[feature_cols]
y = pd.cut(
    df["전체건수"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

categorical_features = ["휴일구분"] 
numeric_features = ["평균기온", "일강수량", "평균상대습도"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

preprocessor = ColumnTransformer( transformers=[ ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features), ("num", StandardScaler(), numeric_features), ] )

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier( n_estimators=100, random_state=42 ),

    "SVM": SVC(kernel="rbf"),
}

results = []
predictions = {}
for name, model in models.items():
    # 전처리 후 아래 삭제 -----------------------------------
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
 
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    #mae = mean_absolute_error(y_test, y_pred)
    #r2 = r2_score(y_test, y_pred)
    f1 = f1_score(y_test,y_pred,average="weighted")
    f1_macro = f1_score(y_test,y_pred,average="macro")
 
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~1500명",
                "1501명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

              precision    recall  f1-score   support

      0~700명       0.81      0.72      0.76       116
   700~1500명       0.32      0.44      0.37        39
    1501명 이상       0.77      0.77      0.77       101

    accuracy                           0.70       256
   macro avg       0.64      0.64      0.63       256
weighted avg       0.72      0.70      0.71       256

              precision    recall  f1-score   support

      0~700명       0.72      0.83      0.77       116
   700~1500명       0.33      0.05      0.09        39
    1501명 이상       0.74      0.86      0.80       101

    accuracy                           0.72       256
   macro avg       0.60      0.58      0.55       256
weighted avg       0.67      0.72      0.68       256

           Model   f1
0  Random Forest 0.71
1            SVM 0.68
